# Fault Tolerance and Elastic Training

## Overview

Handle node failures and dynamic scaling in distributed training.

### Topics
- Checkpoint strategies
- Elastic training with torchrun
- Recovery mechanisms

In [ ]:
import torch
import os
from pathlib import Path

class FaultTolerantCheckpointer:
    """Checkpoint manager with fault tolerance."""
    
    def __init__(self, save_dir, max_checkpoints=3):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_checkpoints = max_checkpoints
    
    def save(self, state, step):
        """Save checkpoint with atomic write."""
        path = self.save_dir / f"ckpt_{step}.pt"
        tmp_path = path.with_suffix('.tmp')
        
        # Write to temp file first
        torch.save(state, tmp_path)
        # Atomic rename
        tmp_path.rename(path)
        
        # Cleanup old checkpoints
        self._cleanup()
    
    def load_latest(self):
        """Load most recent valid checkpoint."""
        ckpts = sorted(self.save_dir.glob("ckpt_*.pt"))
        for ckpt in reversed(ckpts):
            try:
                return torch.load(ckpt)
            except Exception:
                continue
        return None
    
    def _cleanup(self):
        ckpts = sorted(self.save_dir.glob("ckpt_*.pt"))
        for ckpt in ckpts[:-self.max_checkpoints]:
            ckpt.unlink()

## Elastic Training Launch

```bash
# Elastic training: min 2, max 8 nodes
torchrun \
    --nnodes=2:8 \
    --nproc_per_node=8 \
    --rdzv_backend=c10d \
    --rdzv_endpoint=master:29500 \
    train.py
```